# Monte-Carlo SImulations in Python

In [2]:
%load_ext pretty_jupyter 

## Introduction

Monte Carlo simulations are a powerful tool for understanding the behavior of a statistic under various assumptions and random inputs. In this walkthrough, we'll learn how to implement these simulations in Python, specifically focusing on testing the stationarity of a time series with the Augmented Dickey-Fuller (ADF) test.

The problem we aim to address is understanding how a particular statistic behaves under specific conditions. To set the stage, imagine you are conducting an Augmented Dickey-Fuller (ADF) test to determine whether a time series $(z_t)$ is stationary or has a **unit root**.

As an attentive econometrics student, you would recognize that the ADF test involves estimating the following regression model (neglecting trends and augmentation terms for simplicity):

$$
\Delta z_t = \theta + \beta z_{t-1} + \varepsilon_t
$$

Here, the coefficient $\beta$ indicates whether the series $( z_t)$ is stationary $(\beta < 0)$ or non-stationary $(\beta \geq 0)$. The test statistic for $(\beta)$ is calculated as:

$$
t_\beta = \frac{\hat{\beta}}{\text{se}(\hat{\beta})}
$$

to test the null hypothesis $H_0: \beta = 0$ against the alternative $H_A: \beta < 0$.

We must understand the test statistic distribution under the null hypothesis $H_0$ to evaluate this hypothesis. Unfortunately, $ t_\beta$ does not follow a standard distribution (e.g., the t-distribution commonly assumed for regression coefficients).

In this walkthrough, we will demonstrate how to use Python to simulate the actual distribution of this test statistic through a Monte Carlo simulation.

## Monte Carlo simulation-The Idea

When performing the ADF test, you estimate the regression model using a specific time series (e.g., inflation rates) and obtain one test statistic. This single value is insufficient to infer the distribution of the statistic. To make a well-informed decision, you must compare the observed statistic against its distribution under $H_0$.

In practice, applied researchers often rely on theoretical derivations or simulations to understand the behavior of test statistics under the null hypothesis. When deriving the theoretical distribution is infeasible, Monte Carlo simulations serve as a powerful tool.

For the ADF test, the null hypothesis assumes:

$$
\Delta z_t = \theta + 0 \cdot z_{t-1} + \varepsilon_t
$$
or equivalently:
$$
z_t = \theta + 1 \cdot z_{t-1} + \varepsilon_t
$$

This describes a random walk with drift (where the drift is $θ$). 

The Monte Carlo simulation process can be broken down into the following steps:

1. **Define the Data-Generating Process (DGP):** For instance:
   $
   z_t = 0 + 1 \cdot z_{t-1} + \varepsilon_t
   $
   where $θ=0$  and $\varepsilon_t \sim \mathcal{N}(0, 1)$.

2. **Simulate the process \( N \) times:** Generate $N$ datasets based on the DGP defined above.

3. **Apply the ADF test to each dataset:** Calculate and save the test statistic for each simulated dataset.

4. **Analyze the distribution of test statistics:** Use the collection of test statistics to describe the empirical distribution under $H_0$.

This method enables you to simulate the distribution of the test statistic under the null hypothesis, making it possible to compare your observed statistic to this distribution.

> **Note:** This approach assumes that the distribution of the test statistic does not depend on the specific parameter choices in step 1 (e.g., $( \theta = 0)$ and $({Var}(\varepsilon_t) = 1 ))$. If this is true, the test statistic is termed *pivotal*. Proving whether a statistic is pivotal is beyond the scope of this walkthrough.

## Random Processes and Random Variables

In Monte Carlo simulations, we need to simulate random processes $z_t$. To do this, we generate random variables $\varepsilon_t \sim \mathcal{N}(0, 1)$. In Python, you can draw random variables from [many different distributions](https://www.geeksforgeeks.org/random-numbers-in-python/). Python provides several libraries for working with random numbers, the most common being `numpy`. To generate random variables from a standard normal distribution in Python, we use the `numpy.random.normal() function`. Here's how it works:

In [31]:
# Import the numpy library
import numpy as np

# Generate three random numbers from a standard normal distribution
random_numbers = np.random.normal(loc=0, scale=1, size=3)
print(random_numbers)
# Output: [ 0.49671415 -0.1382643   0.64768854 ]


[-0.25142062 -0.86488739  1.8761524 ]


Every time you run this code, you get a new set of random numbers. Let’s demonstrate this by generating another set:

In [34]:
# Generate another set of three random numbers
another_set = np.random.normal(loc=0, scale=1, size=3)
print(another_set)
# Output: [ 1.52302986 -0.23415337 -0.23413696 ]


[-0.30695279 -0.18056779 -0.8767367 ]


So far, the results look random. However, random numbers generated by computers **are not truly random**. They are pseudo-random and depend on a starting point called a `seed`. By setting the seed, you can control the sequence of random numbers. Here’s an example:

In [41]:
# Set a random seed and generate random numbers
np.random.seed(1)
seeded_random_numbers = np.random.normal(loc=0, scale=1, size=3)
print(seeded_random_numbers)

# Generate another set with the same seed
another_seeded_set = np.random.normal(loc=0, scale=1, size=3)
print(another_seeded_set)

[ 1.62434536 -0.61175641 -0.52817175]
[-1.07296862  0.86540763 -2.3015387 ]


If you reset the seed to the same value and generate the numbers again, the results will be **identical**:

In [45]:
# Reset the seed and generate the same sequence of random numbers
np.random.seed(1)
repeat_random_numbers = np.random.normal(loc=0, scale=1, size=3)
print(repeat_random_numbers)

# Generate another identical set
repeat_another_set = np.random.normal(loc=0, scale=1, size=3)
print(repeat_another_set)

[ 1.62434536 -0.61175641 -0.52817175]
[-1.07296862  0.86540763 -2.3015387 ]


This demonstrates that the random numbers don’t appear truly random anymore. For all intents and purposes, random draws after calling the `np.random.seed(1)` command are still random, but the sequence is now reproducible. You can always return to the same sequence of random draws by resetting the seed to the same value (here `1`, but it could be any number).

You can think of this as an infinitely long list of random numbers. When you start Python without setting a seed, it will begin at some random location on this list. By using `np.random.seed()`, you specify a starting point on the list, ensuring you can return to that exact position whenever needed.

In simulation studies, it is standard practice to set a seed at the beginning. This ensures that the results of the simulation can be replicated exactly the next time the code is run, a critical requirement for reproducibility in scientific research.

## Monte Carlo Simulation, Classical Regression Setup

Let’s begin with a simple example where the distribution of a test statistic is well-known. Consider the linear regression model:

$$
y = X \beta + u
$$

Here:
- $X$ is a $( T \times (k+1))$ matrix of regressors (including a constant and $k$ variables).
- $u$ is a $(T \times 1)$ vector of white noise error terms, drawn from a standard normal distribution.
- $T$ is the sample size.

For $k = 1$, this simplifies to:

$$
y = \beta_0 + x \beta_1 + u
$$

or, for individual observations:

$$
y_t = \beta_0 + \beta_1 x_t + u_t
$$

If the regressor $x$ is exogenous, we know that the t-test for $β_0$ testing the null hypothesis $( H_0: β_1 = 0.5)$, follows a $(t)$-distribution with $T - 2$ degrees of freedom. Let’s verify this using a Monte Carlo simulation.

### Steps for Monte Carlo Simulation in Python

1. **Define the Data-Generating Process (DGP):** 
   Define the process $y_t = \beta_0 + \beta_1 x_t + u_t$, where:
   - $β_0 = 0$
   - $β_1 = 0.5$
   - $u_t \sim \mathcal{N}(0,1))$
   - $x_t \sim \text{Uniform}(0,1))$

2. **Simulate the process \( N \) times:**
   Generate $N$ datasets based on the defined DGP.

3. **Apply the t-test for each dataset:**
   Compute the t-test statistic for $β_1$ for each dataset.

4. **Describe the distribution of test statistics:**
   Analyze and compare the simulated test statistics to the theoretical $(t)$-distribution.

[//]: # (-.- .tabset)

#### Step 1 - Specify the Data Generating Process (DGP):


Let’s define the DGP as follows:

$$y = 0 +  x  0.5 + u$$

Where: 
- $T=10$, the sample size.
- $x$ values are drawn from a uniform distribution $x \sim \text{Uniform}(0,1)$
- $u$ values are drawn from a standard normal distribution $u \sim \mathcal{N}(0,1)$


#### Step 2 - Simulate the process N times

In [73]:
import numpy as np
import time

# Parameters for simulation
N = 10000  # Number of simulations
T = 10     # Sample size
beta_0 = 0
beta_1 = 0.5

# Set seed for reproducibility
np.random.seed(123)

# Measure the start time
start_time = time.time()

# Generate random values for x and u
x = np.random.uniform(0, 1, (T, N))  # T x N matrix of Uniform(0,1) values
u = np.random.normal(0, 1, (T, N))  # T x N matrix of Normal(0,1) values

# Generate y values based on the DGP
y = beta_0 + beta_1 * x + u

# Measure the end time
end_time = time.time()

# Calculate the time taken
time_taken = end_time - start_time

# Print the time taken
print(f"Time taken: {time_taken:.6f} seconds")


Time taken: 0.005023 seconds


As we can indicate, this process generated $N$ series, each of length $T$. The values for $x$  are stored in the respective columns of the $x$ matrix, and the corresponding values for $y$ are stored in the columns of the $y$ matrix.  At first glance, it may not be immediately intuitive how we managed to create 10,000 series with just a few lines of code. However, by using Python's vectorized operations with `numpy`, we efficiently generate all the required series simultaneously without the need for explicit loops. This approach leverages Python's ability to handle array-based computations, making it concise and computationally efficient.

However, ou could achieve the same result using a loop, which may initially feel more intuitive to those unfamiliar with vectorized operations. Here's how you could use a loop to generate the same $N$ series:

In [77]:
import numpy as np
import time

# Parameters for simulation
N = 10000  # Number of simulations
T = 10     # Sample size
beta_0 = 0
beta_1 = 0.5

np.random.seed(123)  # Set seed for reproducibility

# Measure start time
start_time = time.time()

# Create storage matrices for x and y
x_loop = np.zeros((T, N))
y_loop = np.zeros((T, N))

# Generate datasets one at a time
for n in range(N):
    x_temp = np.random.uniform(0, 1, T)  # Generate x for one dataset
    u_temp = np.random.normal(0, 1, T)  # Generate u for one dataset
    y_temp = beta_0 + beta_1 * x_temp + u_temp  # Apply DGP
    
    x_loop[:, n] = x_temp  # Store x values in the nth column
    y_loop[:, n] = y_temp  # Store y values in the nth column

# Measure end time
end_time = time.time()

# Calculate the time taken
time_taken = end_time - start_time

# Print the time taken
print(f"Time taken: {time_taken:.6f} seconds")


Time taken: 0.418881 seconds


Going through the loop achieved the same result and may initially be more intuitive for understanding the process. However, it took significantly longer compared to the vectorized approach.

You should avoid using loops in Python for tasks involving large datasets whenever possible. Instead, leveraging vectorized operations, like those provided by `numpy`, can dramatically improve performance and code efficiency.

#### Step 3: Apply Test Statistic
At this stage, we have our simulated data saved in $x$ and $y$ with $N$ datasets stored in the respective columns of the matrices. Now, we need to calculate the test statistic for each dataset. Specifically, we test the null hypothesis:

$$ H_0: β_1 = 0.5$$

against the alternative:

$$H_A: \beta_1 \neq 0.5$$


In [86]:
import statsmodels.api as sm

# Storage for test statistics
t_statistics = []

# Loop through each dataset
for n in range(N):
    # Extract the nth column for x and y
    x_n = x_loop[:, n]
    y_n = y_loop[:, n]
    
    # Add a constant term to x for the intercept
    X_n = sm.add_constant(x_n)
    
    # Fit the regression model
    model = sm.OLS(y_n, X_n).fit()
    
    # Calculate the t-statistic for β1
    beta_1_hat = model.params[1]
    se_beta_1_hat = model.bse[1]
    t_stat = (beta_1_hat - beta_1) / se_beta_1_hat
    
    # Store the t-statistic
    t_statistics.append(t_stat)


So, we begin by extracting the $T$-dimensional vectors $x_n$ and $y_n$ for each column in the $x$ and $y$ matrices. Then, we use the `sm.add_constant()` to include an intercept term in the regression model. Then, we proceed to fit an Ordinary Least Squares regression of $y_n$ and $x_n$ by using `statsmodels.OLS()`. The next phase consists of computing the $ \hat{\beta_1}$ and its standard errors ${\text{se}(\hat{\beta})}$. The t-statistic is calculated as:

$$
t = \frac{\hat{\beta_1}-{\beta_1}}{\text{se}(\hat{\beta_1})}
$$

Following this process, we can store the results by appending the t-statistic to the `t_statistics` list for later analysis.

> Here, it is essential to remember that the test statistic measures how far $ \hat{\beta_1}$ is from the hypothesized value $\beta_1\$ =0.5 standardised by its standard error. Also, the loop processes one dataset at a time, extracting the data and calculating the t-statistic. 

#### Step 4: Describe the distribution

We can describe their distribution now that we have $N$ test statistics. Under the null hypothesis $ H_0: β_1 = 0.5$,  the t-statistics are theoretically $t$-distributed with $T-2$ degrees of freedom, which in this case is $10-2=8$. 

For a two-tailed test at the 5% significance level, we reject $H_0$ if the absolute value of the t-statistic exceeds the critical value of 2.306. This critical value can be obtained using Python.

In [127]:
import scipy.stats as stats

# Degrees of freedom
df = T - 2

# Calculate the critical value for a two-tailed test at 5% significance level
critical_value = stats.t.ppf(0.975, df)
print(f"Critical value: {critical_value:.3f}")

# Calculate the rejection rate
rejection_rate = np.mean(np.abs(t_statistics) > critical_value)
print(f"Rejection rate: {rejection_rate:.4f}")


Critical value: 2.306
Rejection rate: 0.0493


Here, we are basically using `stats.t.ppf(0.975, df)` calculate the 97.5th percentile of the of the $t$- distribution ( because it is a two tailed test with 2.5% in each tail).

Now we can check how many simulated t-tests reject $H_0$ and calculate the rejection rate:

In [132]:
# Calculate the rejection rate
rejection_rate = np.mean(np.abs(t_statistics) > critical_value)
print(f"Rejection rate: {rejection_rate:.4f}")

Rejection rate: 0.0493


Here, the `np.abs(t_statistics)` computes the absolute values of the simulated t-statistics. We perform a two-tailed test and care about deviations in both directions. For each t-statistic, check if $∣𝑡∣$ > critical value. Then, we can calculate the proportion of t-statistics that exceed the critical value using `np.mean()`. Since $H_0$ is true for all simulated processes, approximately 5% of tests should incorrectly reject $H_0$ at the 5% significance level.

The rejection rate $0.0493$ indicates that approximately $4.93%$ of the simulated t-tests incorrectly rejected the null hypothesis $H_0$, which is very close to the theoretical expectation of $5%$. This slight deviation is due to random variation inherent in the simulation process, especially with $N=10000$. 

We can also visualize the empirical and theoretical distributions of the t-statistics using `lets-plot` for visualization:

In [144]:
import numpy as np
import pandas as pd
from scipy.stats import t
from lets_plot import *

# Initialize Lets-Plot for Jupyter
LetsPlot.setup_html()

# Parameters
df = T - 2  # Degrees of freedom
x_vals = np.linspace(-5, 5, 500)  # Range for theoretical t-distribution

# Convert t-statistics to a pandas DataFrame for plotting
data = pd.DataFrame({'t_statistics': t_statistics})

# Calculate the theoretical t-distribution density
theoretical_density = t.pdf(x_vals, df=df)

# Plot empirical and theoretical densities
plot = (
    ggplot(data, aes(x='t_statistics')) +
    geom_density(color='black', size=2, alpha=0.7, label="Empirical Density") +
    geom_line(aes(x=x_vals, y=theoretical_density), color='blue', size=2, label="Theoretical t(8) Density") +
    ggtitle("Empirical vs Theoretical t-Distribution") +
    xlab("t-Statistic") +
    ylab("Density")
)

# Show the plot
plot.show()


We call the `data['t_statistics'].plot.kde(ind=np.linspace())`function and specify "`ind=np.linspace(-5, 5, 500))`". This function Calculates the kernel density estimate (KDE) of the simulated $t-statistics$ over a range of values from $-5 to5$. Then, we use the `t.pdf(x_vals, df=df)` to compute the probability density function (PDF) of the theoretical $t-distribution$ with $df=8$ degrees of freedom. The `geom_density()` generates a smooth curve based on the simulated t-statistics, and the `geom_line()` overlays the theoretical t-distribution.

You can see that the simulated test distribution is very close to the theoretical distribution. The empirical density (black line) aligns closely with the theoretical $𝑡(8)$-distribution (blue line), indicating that the Monte Carlo simulation accurately replicates the theoretical behaviour of the 
$t-statistic$ under the null hypothesis. This confirms the validity of the simulation process and demonstrates that the test is well-calibrated.


[//]: # (-.- .unlisted .unnumbered)

## Extension: Additional Questions to Explore

To deepen your understanding of the Monte Carlo simulation and hypothesis testing, consider exploring the following extensions:

- 1. **Compare the Distribution to a Normal Distribution**
     - While the $t-statistic$ is theoretically $t-distributed$ with $T-2$ degrees of freedom, how does it compare to a standard normal distribution?
     - Modify the code to overlay the PDF of a standard normal distribution $N(0,1)$ on the same plot for comparison.

In [155]:
# Generate the standard normal density
normal_density = stats.norm.pdf(x_vals, loc=0, scale=1)  # Standard normal distribution

# Add the standard normal distribution to the plot
plot = (
    ggplot(data, aes(x='t_statistics')) +
    geom_density(color='black', size=2, alpha=0.7, label="Empirical Density") +
    geom_line(aes(x=x_vals, y=theoretical_density), color='blue', size=2, label="Theoretical t(8) Density") +
    geom_line(aes(x=x_vals, y=normal_density), color='red', size=2, label="Standard Normal Density") +
    ggtitle("Empirical vs Theoretical vs Normal Distributions") +
    xlab("t-Statistic") +
    ylab("Density")
)

plot.show()

**Questions to Consider:**

- Does the empirical t-distribution closely resemble the standard normal distribution?
- How do the tails of the 𝑡(8)-distribution differ from the normal distribution?

- 2. **Parameter Dependency**
     - Investigate whether your conclusions depend on the values of $β_0$ or the error variance $Var(u)$.
     - Modify the simulation with different values of $β_0$ (e.g., 1 or -1) or $Var(u)$ (e.g., 0.5 or 2).

[//]: # (-.- .tabset)

##### Key Steps

- Change $β_0$ in the DGP: $y=β_0+β_1$x+$u$.
- Change the variance of $u$: Use `np.random.normal(0, std_dev, size)` where `std_dev` is the square root of the new variance.


##### Questions to Consider:

- Does the shape of the empirical distribution or rejection rate change?
- Is the test statistic $ t-distributed$  regardless of these parameter changes?

[//]: # (-.- .unlisted .unnumbered)

- 3. **Pivotal Statistic**
     - Explore whether the test statistic is pivotal, meaning its distribution does not depend on the underlying parameter values of $β_0$ or the error variance $Var(u)$.
     - This can be verified by running simulations with different $β_0$ or $Var(u)$ and comparing the resulting distributions of the $t-distributions$.

**Questions to Consider**:

- Does the distribution of t-statistics remain the same across different parameter values?
- If yes, the statistic is pivotal; if not, it depends on the parameter values.

- 4. **Dependence on the Distribution of $𝑥$**
     - Examine whether the conclusions depend on the distribution that simulates $𝑥$ (e.g., uniform, normal, exponential).
     - Replace `np.random.uniform (0, 1, T)` with other distributions, such as:
       - `np.random.normal(0, 1, T)` (standard normal)
       - `np.random.exponential(1, T)` (exponential distribution)

**Questions to Consider**:

- How does changing the distribution of $𝑥$ affect the empirical distribution of the  𝑡-statistic?
- Does the test remain well-calibrated for different distributions of $𝑥$? 


## Monte-Carlo Simulation, Dickey-Fuller Test Distributions
### Background

The simplest non-stationary process integrated with order one is:

$$y_t=y_t-1+ε_t$$

This is called a **random walk**, and it implies:

$$Δy_t=ε_t$$


To simulate random walks, we:

- Generate a matrix of error terms ($ε_t$) from a standard normal distribution.
- Compute the cumulative sum (`cumsum`) of these errors for each column, simulating the random walks.

In [182]:
import numpy as np
import pandas as pd
from lets_plot import *

# Initialize Lets-Plot for Jupyter
LetsPlot.setup_html()

# Parameters
N = 10  # Number of random walks
T = 12610  # Length of each random walk

np.random.seed(1234)  # Set seed for reproducibility

# Generate error terms (T x N matrix)
error_terms = np.random.normal(0, 1, (T, N))

# Compute random walks by cumulative sum along rows
random_walks = np.cumsum(error_terms, axis=0)

# Convert to a DataFrame for plotting
df = pd.DataFrame(random_walks, columns=[f"RW{i+1}" for i in range(N)])

# Plot random walks
plot = (
    ggplot(df.reset_index().melt(id_vars='index', var_name='Random Walk', value_name='Value'), aes(x='index', y='Value', color='Random Walk')) +
    geom_line() +
    ggtitle("Examples of Simulated Random Walks") +
    xlab("Time") +
    ylab("Value")
)
plot.show()


How do we simulate error terms? We can create a $T × N$ matrix of error terms drawn from a standard normal distribution $N(0,1)$ by calling the `np.random.normal(0, 1, (T, N))` function. Then, we can generate random walks by computing the cumulative sum of error terms along each column and running the `np.cumsum(error_terms, axis=0)` function. After these necessary steps, we can plot the random walks by preparing the data for plotting multiple time series. For this purpose, we use the `reset_index()` and `melt()` functions, whereas `geom_line()` creates a line plot, with each random walk represented as a separate line.

As you can see, the plot displays 10 random walks (lines), each representing a simulated time series of length **$T=12610$**. Each random walk starts at $y_0=0$ and evolves by summing up the random errors $ε_t$. As you can indicate, the entire simulation is done without loops, leveraging `numpy's` efficient array operations. Also, the seed (`np.random.seed(1234)`) ensures the same random walks are generated each time the code is run.

As it turns out, many economic processes possess characteristics unlike those of a random walk. Take, for example, the **Pound/USD exchange rate**. To  fetch and plot the GBP/USD exchange rate in Python, you can use the `pandas_datareader` library to access exchange rate data, such as from the Federal Reserve Economic Data (FRED) database and `lets-plot` for visualization.

In [188]:
import pandas as pd
import pandas_datareader.data as web
from lets_plot import *

# Initialize Lets-Plot for Jupyter
LetsPlot.setup_html()

# Fetch GBP/USD exchange rate data
start_date = "1965-01-01"
end_date = "2024-11-18"
gbp_usd = web.DataReader("DEXUSUK", "fred", start_date, end_date)

# Drop NaN values
gbp_usd.dropna(inplace=True)

# Prepare the data for plotting
gbp_usd.reset_index(inplace=True)
gbp_usd.columns = ['Date', 'Exchange Rate']

# Plot the GBP/USD exchange rate
plot = (
    ggplot(gbp_usd, aes(x='Date', y='Exchange Rate')) +
    geom_line(color='blue', size=1) +
    ggtitle("GBP to USD Exchange Rate") +
    xlab("Date") +
    ylab("Exchange Rate")
)
plot.show()


> The `"DEXUSUK"` fucntion here is the FRED code for the GBP/USD exchange rate. Then the `web.DataReader()` fucntion retrieves the data for the specified time period (`start_date to end_date`).

The variation observed in the exchange rate closely resembles the behavior of the simulated random walks. Interestingly, if we use one random walk as an explanatory variable to explain the exchange rate in a regression, the relationship might appear statistically significant, even though no true relationship exists.

To illustrate this, we will run regressions where the exchange rate is the dependent variable, and each of the 10 simulated random walks is used as an explanatory variable. This demonstrates why we simulated random walks with  $𝑇=$12610, ensuring they have the same length as the exchange rate series. After running these regressions, we will save and examine the t-statistics for the slope coefficients.

In [192]:
import numpy as np
import statsmodels.api as sm
import pandas as pd

# Simulate random walks
N = 10  # Number of random walks
T = 12610  # Length of each random walk
np.random.seed(1234)
error_terms = np.random.normal(0, 1, (T, N))
random_walks = np.cumsum(error_terms, axis=0)  # Simulate random walks

# Fetch GBP/USD exchange rate data
import pandas_datareader.data as web

start_date = "1965-01-01"
end_date = "2024-11-18"
gbp_usd = web.DataReader("DEXUSUK", "fred", start_date, end_date).dropna()
gbp_usd_values = gbp_usd.values.flatten()  # Convert to 1D array

# Ensure lengths match
gbp_usd_values = gbp_usd_values[:T]

# Storage for t-statistics
t_statistics = []

# Loop through each random walk
for j in range(N):
    x = random_walks[:, j]  # Use the j-th random walk as the regressor
    y = gbp_usd_values      # Use GBP/USD as the dependent variable
    
    # Add constant for the regression
    X = sm.add_constant(x)
    
    # Fit OLS model
    model = sm.OLS(y, X).fit()
    
    # Extract t-statistic for the slope (2nd coefficient)
    t_stat = model.tvalues[1]
    t_statistics.append(t_stat)

# Convert to DataFrame for better display
t_stats_df = pd.DataFrame({'Random Walk': [f'RW{i+1}' for i in range(N)],
                           't-Statistic': t_statistics})

# Print the results
print(t_stats_df)


  Random Walk  t-Statistic
0         RW1    69.632851
1         RW2    68.412025
2         RW3     9.771741
3         RW4   -28.401854
4         RW5   -50.002561
5         RW6   -20.181343
6         RW7    87.523243
7         RW8    -5.429028
8         RW9   -12.888205
9        RW10   -26.426246


Here, we are calling the `np.cumsum()` function to simulate $N=10$ random walks of length $T=12610$. Using the `pandas_datareader` we download the GBP/USD exchange rate data and ensure the exchange rate series has the same length as the random walks by truncating it to $T=12610$. We then run regressions ensuring for each random walk:
- Use it as the explanatory variable ($x$).
- Use the exchange rate series ($y$) as the dependent variable.
- Fit an OLS regressio using `statsmodels.OLS()`.

Then, save the t-statistic from the slope coefficient (2nd coefficient) from each regression into a list and convert the t-statistics into a `pandas.DataFrame` for easier visualization. 

We now have 10 t-statistics, one for each regression of the exchange rate on the 10 simulated random walks. You would be correct if you noted that these results are problematic because the coefficient standard errors do not account for residual autocorrelation. However, using robust standard errors would not solve the underlying issue here.

The key takeaway is that in 9 of the 10 regressions, the t-statistics are extremely large (in absolute value), which would lead us to conclude that the relationships between the exchange rate and these random walks are statistically significant. Yet, we know these relationships are **spurious** because:

1. The exchange rate and random walks are unrelated in reality.
2. The random walks were generated entirely randomly.
 
This happens because we are regressing one non-stationary series (exchange rate) on another (random walks). Standard inference based on the Central Limit Theorem (asymptotic normality) is invalid for non-stationary series. This often leads to significant results even when no real relationship exists.


### Simulating ADF-test distributions

The Augmented Dickey-Fuller (ADF) test is one of the most widely used tests to determine whether a series is stationary or non-stationary. It performs a t-test to evaluate the null hypothesis:


$$H_0: \alpha = 0$$

against the alternative:

$$H_A: \alpha < 0$$

in the following regression:

$$ \Delta y_t = \alpha y_{t-1} + \epsilon_t$$

[//]: # (-.- .tabset)


#### Simulating the ADF Test Statistic

The ADF test statistic is calculated as:

$$
t_\alpha = \frac{\hat{\alpha} - 0}{\text{se}(\hat{\alpha})}
$$

The null hypothesis $$H_0: \alpha = 0$$ implies:

$$
\Delta y_t = \epsilon_t
$$

This describes a random walk, which is a non-stationary process. If $\alpha < 0 $, the process is stationary.

To simulate the distribution of the ADF test statistic:
1. Simulate $N$ processes under the null hypothesis.
2. Calculate the test statistic for each simulated process.
3. Compare the distribution of the simulated test statistics to the standard normal distribution.

#### Deterministic Terms in ADF Test

The ADF test can take into account different deterministic components. Two common specifications are:

1. With a constant:
   $
   \Delta y_t = \delta + \alpha y_{t-1} + \epsilon_t
   $

2. With a constant and a trend:
   $
   \Delta y_t = \delta + \gamma t + \alpha y_{t-1} + \epsilon_t
   $

Regardless of the specification, the relevant test is always the t-test for \( t_\alpha \).

#### Note on Residual Autocorrelation

In practice, lagged terms of $\Delta y_t$ are often added to the regression to account for residual autocorrelation. These extended models are referred to as the Augmented Dickey-Fuller tests. However, in the context of this simulation, lagged terms are not necessary, as the focus is on the simplest case.



In [201]:
import numpy as np

# Parameters
N = 10000  # Number of simulations
T = 1000   # Length of each series

# Set seed for reproducibility
np.random.seed(1234)

# Step 1: Generate error terms and simulate random walks
data = np.random.normal(0, 1, (T, N))  # T x N matrix of error terms
y0 = np.cumsum(data, axis=0)  # Compute cumulative sum to simulate random walks

# Step 2: Create matrices for Δy_t (explained) and y_{t-1} (explanatory)
yy = y0[1:, :] - y0[:-1, :]  # Δy_t: Differences between consecutive rows
xx = y0[:-1, :]              # y_{t-1}: Values shifted by one time step

In this simulation, we generate $N=10,000$ random walks, each of length $T=1,000$ by first creating a matrix of random error terms drawn from a standard normal distribution. Using the cumulative sum of these errors, we simulate the random walk for each column. To prepare the data for the Dickey-Fuller regression, we calculate the differences ($Δ𝑦_𝑡$) between consecutive time points as the explained variable and extract the lagged values $y_{t-1}$ as the explanatory variable. This results in two matrices: $Δ𝑦_𝑡$ and $y_{t-1}$, both of size $(T-1)×N$ Ready for performing the ADF test across all simulations.    

[//]: # (-.- .unlisted .unnumbered)

With these matrices, you can now loop through the $𝑁$ columns, fit the Dickey-Fuller regression, and calculate the test statistic for each simulated process:

In [214]:
import statsmodels.api as sm

# Storage for test statistics
tau_statistics = []

# Loop through each simulated process (each column of yy and xx)
for i in range(N):
    y = yy[:, i]  # Extract Δy_t (explained variable)
    x = xx[:, i]  # Extract y_{t-1} (explanatory variable)
    
    # Add a constant term to the explanatory variable
    X = sm.add_constant(x)
    
    # Fit the OLS regression
    model = sm.OLS(y, X).fit()
    
    # Extract the t-statistic for the slope (coefficient of y_{t-1})
    t_stat = model.tvalues[1]
    tau_statistics.append(t_stat)


To calculate the ADF test statistic for each simulated process, we loop through the $N=10,000$ columns of the matrices $Δ𝑦_𝑡$ (explained variable) and $y_{t-1}$ (explanatory variable).  For each column, we add a constant term to $y_{t-1}$ and fit an Ordinary Least Squares (OLS) regression of $Δ𝑦_𝑡$ on $y_{t-1}$ and the constant. 

The t-statistic for the slope coefficient of $y_{t-1}$ which tests the null hypothesis that the series is non-stationary ($H_0: \alpha = 0$),  is extracted and stored. This process provides a collection of $𝑁$ ADF test statistics, which can be analyzed to assess their empirical distribution and compare it to theoretical expectations.

### Next Steps

With the simulated ADF test statistics ready, we can:

- Analyze their distribution and compare it to the critical values of a standard normal or Dickey-Fuller test distribution.
- Calculate empirical rejection rates at different significance levels.

[//]: # (-.- .tabset)


#### Step 1: Analyze the Distribution of Simulated ADF Test Statistics

We will visualize the distribution of the simulated ADF test statistics and compare it with:

- The critical values from a standard normal distribution.
- The critical values from a Dickey-Fuller distribution (specific to the test).

In [225]:
import pandas as pd
from lets_plot import *
from scipy.stats import norm

# Initialize Lets-Plot for Jupyter
LetsPlot.setup_html()

# Convert ADF test statistics to a DataFrame for plotting
data = pd.DataFrame({'ADF Test Statistic': tau_statistics})

# Calculate the standard normal distribution for comparison
x_vals = np.linspace(-5, 5, 500)  # Define x-axis values
normal_density = norm.pdf(x_vals)  # Standard normal density values

# Define the 5% critical value for the Dickey-Fuller test
critical_value = -2.86  # Example critical value

# Plot the empirical histogram, standard normal curve, and critical value
plot = (
    ggplot(data, aes(x='ADF Test Statistic')) +
    geom_histogram(binwidth=0.2, color='black', fill='gray', alpha=0.5, label='Simulated ADF Statistics') +
    geom_line(aes(x=x_vals, y=normal_density), color='blue', size=2, label='Standard Normal Distribution') +
    geom_vline(xintercept=critical_value, color='red', linetype='dashed', size=2, label='5% Critical Value (Dickey-Fuller)') +
    ggtitle("Empirical Distribution of Simulated ADF Test Statistics") +
    xlab("ADF Test Statistic") +
    ylab("Density")
)

plot.show()


This plot confirms that the simulated ADF test statistics align well with the theoretical expectations. The deviations in the tails highlight the need to use the Dickey-Fuller critical values, rather than standard normal critical values, for proper inference. This demonstrates the importance of correctly simulating the distribution of the ADF test statistic when dealing with non-stationary processes.

#### Step 2: Calculate Empirical Rejection Rates
We can calculate the proportion of simulated ADF test statistics that are less than the critical values (e.g., at 1%, 5%, and 10% significance levels) to determine the rejection rates under the null hypothesis.

In [228]:
# Define critical values for the Dickey-Fuller test
critical_values = {"1%": -3.43, "5%": -2.86, "10%": -2.57}

# Calculate rejection rates
rejection_rates = {level: np.mean(tau_statistics < value) for level, value in critical_values.items()}

# Print rejection rates
for level, rate in rejection_rates.items():
    print(f"Rejection rate at {level} significance level: {rate:.4f}")


Rejection rate at 1% significance level: 0.0114
Rejection rate at 5% significance level: 0.0542
Rejection rate at 10% significance level: 0.1054


The rejection rates are very close to the nominal significance levels (1%, 5%, and 10%). This indicates that the simulated ADF test statistics are well-calibrated and match theoretical expectations under the null hypothesis ($H_0: \alpha = 0$). These results validate the correctness of the simulated ADF test statistics and demonstrate that the test performs as expected under the null hypothesis. The rejection rates align closely with the theoretical significance levels, confirming that the Monte Carlo simulation was implemented successfully.

[//]: # (-.- .unlisted .unnumbered)

# Summary

In this walkthrough, we showcased the value of Monte Carlo simulations using Python. First, we utilized the Monte Carlo technique to replicate the distribution of a test statistic for which the theoretical distribution is well-established—specifically, the $𝑡-distribution$. This allowed us to validate our simulation methodology by comparing the empirical and theoretical distributions.

Next, we extended the technique to simulate the distribution of the Augmented Dickey-Fuller (ADF) test statistics, a case where the actual distribution is not standard but follows the Dickey-Fuller distribution. By simulating  $𝑁=10,000$ processes, we replicated the empirical distribution of the ADF test statistics and compared it to the standard normal and Dickey-Fuller critical values. The rejection rates at different significance levels (1%, 5%, and 10%) confirmed that the test statistics are well-calibrated and consistent with theoretical expectations.

Through this process, we demonstrated how Monte Carlo simulations can be applied to understand and replicate the behavior of test statistics under various null hypotheses, particularly in econometrics, where analytical solutions may not always be available. This approach underscores the power and flexibility of Python for advanced statistical analysis and simulation.

This exercise is part of the [ECLR](https://datasquad.github.io/ECLR/) page.